In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip -oq "/content/drive/MyDrive/MyDrive WorkerSafety25 /WorkerSafety25.v1-baseline-v1-no-augmentation.yolov8.zip" -d "/content/WorkerSafety25"

In [ ]:
from pathlib import Path

root = Path("/content/WorkerSafety25")

for split in ["train", "valid", "test"]:
    images = list((root / split / "images").glob("*"))
    print(f"{split}: {len(images)} images")


train: 6500 images
valid: 1156 images
test: 660 images


In [ ]:
!cat "/content/WorkerSafety25/data.yaml"


train: ../train/images
val: ../valid/images
test: ../test/images

nc: 9
names: ['gloves on', 'hardhat', 'ladder', 'no glove', 'no hardhat', 'no safety vest', 'person', 'safety vest', 'undefined']

roboflow:
  workspace: abdullah-albugami
  project: workersafety25-ukkoq
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/abdullah-albugami/workersafety25-ukkoq/dataset/1

In [ ]:
from pathlib import Path
from collections import Counter

names = [
    'gloves on', 'hardhat', 'ladder', 'no glove',
    'no hardhat', 'no safety vest', 'person',
    'safety vest', 'undefined'
]

counts = Counter()

for label_file in Path("/content/WorkerSafety25").glob("*/labels/*.txt"):
    for line in label_file.read_text().splitlines():
        if line.strip():
            class_id = int(line.split()[0])
            counts[class_id] += 1

for class_id, class_name in enumerate(names):
    print(f"{class_id} - {class_name}: {counts[class_id]} labels")

0 - gloves on: 3625 labels
1 - hardhat: 2858 labels
2 - ladder: 2309 labels
3 - no glove: 2121 labels
4 - no hardhat: 2839 labels
5 - no safety vest: 1905 labels
6 - person: 4199 labels
7 - safety vest: 4629 labels
8 - undefined: 155 labels


# 1. Setup: imports and dataset paths


In [ ]:
from pathlib import Path
from collections import Counter
import random
import shutil

SOURCE = Path("/content/WorkerSafety25")
OUT = Path("/content/WorkerSafety25_clean_5cls")

# 2. Define the five target classes
We retain only person, helmet, and safety-vest related classes.

In [ ]:
NEW_CLASS_NAMES = [
    "person",
    "hardhat",
    "no hardhat",
    "safety vest",
    "no safety vest"
]

# old class ID -> new class ID
CLASS_MAP = {
    6: 0,  # person
    1: 1,  # hardhat
    4: 2,  # no hardhat
    7: 3,  # safety vest
    5: 4,  # no safety vest
}

# 3. Filter labels and collect eligible images
Images with none of the five retained classes are excluded.

In [ ]:
samples = []
new_label_counts = Counter()
dropped_images = 0

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for old_split in ["train", "valid", "test"]:
    image_dir = SOURCE / old_split / "images"
    label_dir = SOURCE / old_split / "labels"

    for image_path in image_dir.iterdir():
        if image_path.suffix.lower() not in image_extensions:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"
        kept_lines = []

        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()

                # Correct YOLO format: class x_center y_center width height
                if len(parts) != 5:
                    continue

                old_id = int(parts[0])

                if old_id in CLASS_MAP:
                    new_id = CLASS_MAP[old_id]
                    kept_lines.append(" ".join([str(new_id)] + parts[1:]))
                    new_label_counts[new_id] += 1

        if kept_lines:
            samples.append((image_path, kept_lines))
        else:
            dropped_images += 1

print(f"Images retained: {len(samples)}")
print(f"Images excluded: {dropped_images}")

Images retained: 5468
Images excluded: 2848


# 4. Rebuild the dataset split
A fixed random seed ensures the split is reproducible.


In [ ]:
if OUT.exists():
    raise FileExistsError(
        "The cleaned dataset folder already exists. "
        "Do not rerun this cell unless you intentionally use a new output folder."
    )

random.Random(42).shuffle(samples)

total = len(samples)
n_train = int(total * 0.70)
n_valid = int(total * 0.20)

splits = {
    "train": samples[:n_train],
    "valid": samples[n_train:n_train + n_valid],
    "test": samples[n_train + n_valid:]
}

for split_name, split_samples in splits.items():
    images_out = OUT / split_name / "images"
    labels_out = OUT / split_name / "labels"

    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    for image_path, label_lines in split_samples:
        shutil.copy2(image_path, images_out / image_path.name)

        new_label_file = labels_out / f"{image_path.stem}.txt"
        new_label_file.write_text("\n".join(label_lines) + "\n")

for split_name, split_samples in splits.items():
    percentage = len(split_samples) / total * 100
    print(f"{split_name}: {len(split_samples)} images ({percentage:.1f}%)")

train: 3827 images (70.0%)
valid: 1093 images (20.0%)
test: 548 images (10.0%)


# 5. Create the YOLO data configuration file

In [ ]:
yaml_text = """path: /content/WorkerSafety25_clean_5cls
train: train/images
val: valid/images
test: test/images

names:
  0: person
  1: hardhat
  2: no hardhat
  3: safety vest
  4: no safety vest
"""

(OUT / "data.yaml").write_text(yaml_text)

print("data.yaml created successfully.")

data.yaml created successfully.


 # 6. Verify the cleaned dataset

In [ ]:
print("Final label distribution:\n")

for class_id, class_name in enumerate(NEW_CLASS_NAMES):
    print(f"{class_id} - {class_name}: {new_label_counts[class_id]} labels")

print("\nDataset location:")
print(OUT)

Final label distribution:

0 - person: 4166 labels
1 - hardhat: 2827 labels
2 - no hardhat: 2839 labels
3 - safety vest: 4614 labels
4 - no safety vest: 1905 labels

Dataset location:
/content/WorkerSafety25_clean_5cls


In [ ]:
from pathlib import Path

root = Path("/content/WorkerSafety25_clean_5cls")
total_images = 0

for split in ["train", "valid", "test"]:
    image_count = len(list((root / split / "images").glob("*")))
    label_count = len(list((root / split / "labels").glob("*.txt")))
    total_images += image_count

    print(f"{split}: {image_count} images | {label_count} label files")

print(f"\nTotal images: {total_images}")

for split in ["train", "valid", "test"]:
    image_count = len(list((root / split / "images").glob("*")))
    print(f"{split}: {image_count / total_images:.1%}")


train: 3827 images | 3827 label files
valid: 1093 images | 1093 label files
test: 548 images | 548 label files

Total images: 5468
train: 70.0%
valid: 20.0%
test: 10.0%


In [ ]:
import shutil
from pathlib import Path


zip_path = shutil.make_archive(
    "/content/WorkerSafety25_clean_5cls",
    "zip",
    "/content",
    "WorkerSafety25_clean_5cls"
)


drive_path = Path(
    "/content/drive/MyDrive/WorkerSafety25_clean_5cls_70-20-10.zip"
)

shutil.copy2(zip_path, drive_path)

print("تم الحفظ في Google Drive:")
print(drive_path)